# 5i Benchmark CPU and GPU

Measure a bounded workload, separate transfer and compute time, and learn why a tiny example may not benefit from a GPU.

**Budget:** 10–15 minutes; optional compatible CUDA GPU; a CPU and a few thousand images are enough. No Klique endpoint or token is needed for the default path.

**Before you start:** complete the [JupyterLab introduction](05_Running_Jupyter_Lab.ipynb), prepare Python 3.10+ and the packages in the [support guide](README.md), and select that kernel. Run every cell from a fresh kernel in order. The default is offline: cached real datasets are used when present; otherwise the helper explicitly identifies an 8×8 scikit-learn digits fallback. That fallback is **not MNIST, EMNIST, or Fashion-MNIST**, and its metrics are not comparable to theirs. Downloads are opt-in; read [dataset sources and terms](DATASETS.md) first.

Save your work in the approved persistent workspace. No live Klique launch or restore is established by these local exercises.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
LAB = HERE if (HERE / "image_lab.py").is_file() else HERE / "05_Jupyter"
if not (LAB / "image_lab.py").is_file():
    raise RuntimeError("Start this kernel in the repository root or 05_Jupyter folder.")
if str(LAB) not in sys.path:
    sys.path.insert(0, str(LAB))
from image_lab import load_image_data, train_baseline
SEED = 42
OUTPUT = LAB / "lab_outputs"
OUTPUT.mkdir(exist_ok=True)


## A useful CPU-only run

NumPy provides a CPU baseline even without PyTorch. Warm up first and use several repetitions. Keep the matrix size fixed and report the median. These wall-clock times depend on load, thread settings, BLAS implementation, dtype, and hardware. They are observations from this session, not universal claims.


In [ ]:
from time import perf_counter
import statistics
import platform
rng = np.random.default_rng(SEED)
size = 512
left = rng.standard_normal((size, size)).astype(np.float32)
right = rng.standard_normal((size, size)).astype(np.float32)
_ = left @ right
cpu_times = []
for _ in range(5):
    start = perf_counter(); product = left @ right
    cpu_times.append(perf_counter() - start)
print({"backend": "numpy", "dtype": "float32", "matrix_size": size,
       "median_seconds": statistics.median(cpu_times),
       "python": platform.python_version(), "numpy": np.__version__})
assert np.isfinite(product).all()


## Optional apples-to-apples PyTorch comparison

Install the approved PyTorch build for this environment. This block uses the same PyTorch operation, shape, values, and float32 dtype on CPU and optionally CUDA. It warms each device, synchronizes CUDA around timings, reports one transfer separately, and checks approximate numerical agreement. It excludes data loading and is **not** a training-throughput or whole-job benchmark.

Leave `BENCHMARK_CUDA = False` unless the GPU queue and compatibility are approved. This notebook never launches a GPU instance. A GPU can be slower for small workloads because overhead dominates. Do not compare the NumPy timing directly with PyTorch and call that a device-only speedup.


In [ ]:
BENCHMARK_CUDA = False
try:
    import torch
except ImportError:
    torch = None
    print("PyTorch is not installed; NumPy CPU results above are the completed baseline.")
if torch is not None:
    devices = ["cpu"]
    if BENCHMARK_CUDA:
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA requested but unavailable. Check the approved image, driver, and allocation.")
        devices.append("cuda")
    torch.backends.cuda.matmul.allow_tf32 = False
    def sync(device):
        if device == "cuda": torch.cuda.synchronize()
    rows = []
    for device in devices:
        sync(device); transfer_start = perf_counter()
        a = torch.from_numpy(left).to(device); b = torch.from_numpy(right).to(device)
        sync(device); transfer_seconds = perf_counter() - transfer_start
        for _ in range(2): _ = a @ b
        sync(device)
        times = []
        for _ in range(5):
            sync(device); start = perf_counter(); out = a @ b; sync(device)
            times.append(perf_counter() - start)
        np.testing.assert_allclose(out.cpu().numpy(), product, rtol=1e-3, atol=1e-3)
        rows.append({"device": device, "median_compute_seconds": statistics.median(times),
                     "initial_transfer_seconds": transfer_seconds, "torch": torch.__version__,
                     "cpu_threads": torch.get_num_threads(), "matrix_size": size, "dtype": "float32"})
        del a, b, out
    print(rows)


## Checkpoint, exercise, and cleanup

**Checkpoint:** You have a warm CPU measurement and, only when explicitly enabled, synchronized comparable PyTorch CPU/CUDA timings with numerical checks.

**Try it:** Repeat at sizes 256 and 1024, leaving dtype and repetitions fixed. Note whether transfer overhead changes the conclusion and keep the compute budget small.

**Troubleshooting:** if imports fail, select the kernel where the section requirements were installed. If a data host is blocked, keep downloads off and use the named offline fixture, or ask your administrator to stage the official data in the cache. Do not disable TLS verification. A convergence warning means the solver hit its iteration budget; record it rather than silently claiming an optimum.

Save the notebook and useful artifacts under `lab_outputs/`; it is ignored by Git but may still enter workspace snapshots. Clear outputs before sharing if you used private data. Shut down unused kernels, then stop the managed session and verify its stopped state as in the introduction. Closing a browser tab is not cleanup.

**Next:** [5j: Create an Experiment Report](05j_Create_an_Experiment_Report.ipynb) · **Index:** [Jupyter labs](README.md) · [Orientation](../00_Orientation/00_Orientation.ipynb)
